# Análise Exploratória dos Dados

## Dicionário de dados

In [1]:
# importar os pacotes necessários
import pandas as pd
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import warnings
import os
warnings.filterwarnings("ignore")

import numpy as np

In [2]:
dicionario = pd.read_csv("../data/raw/dicionario_dados.csv", sep=";")
dicionario

,base,coluna,descricao
0,base_cadastral,id_cliente,Identificador único do cliente
1,base_cadastral,sexo,Sexo do cliente (M ou F)
2,base_cadastral,data_nascimento,Data de nascimento do cliente
3,base_cadastral,qtd_filhos,Quantidade de filhos declarados
4,base_cadastral,qtd_membros_familia,Quantidade total de membros da família
...,...,...,...
64,historico_parcelas,numero_parcela,Número sequencial da parcela dentro do contrato
65,historico_parcelas,data_prevista_pagamento,Data inicialmente prevista para o pagamento da...
66,historico_parcelas,data_real_pagamento,Data real em que o pagamento foi efetuado
67,historico_parcelas,valor_previsto_parcela,Valor original previsto da parcela


## Funções

In [3]:
def agg_hist_contract(hist_contratos, hist_parcelas):
    # unir historico de contratos com parcelas
    hist_total = hist_contratos.merge(
        hist_parcelas, 
        how="left", 
        on=["id_contrato", "id_cliente"]
    )

    hist_total["dias_atraso_parcela"] = (pd.to_datetime(hist_total["data_real_pagamento"]) - pd.to_datetime(hist_total["data_prevista_pagamento"])).dt.days

    hist_total = hist_total.fillna(-1)

    # CRIACAO DA VARIÁVEL ALVO DE ACORDO COM O REGULADOR (BACEN)
    hist_total["target"] = np.where(
        hist_total["dias_atraso_parcela"] > 90,
        1,
        0
    )

    features_temp = hist_total.groupby(
        [
            "id_cliente",
            "id_contrato", 
            "tipo_contrato", 
            "status_contrato", 
            "data_decisao", 
            "valor_credito", 
            "valor_bem",
            "valor_parcela",
            # "percentual_entrada",
            # "qtd_parcelas_planejadas",
            "tipo_pagamento",
            "finalidade_emprestimo",
            "tipo_cliente",
            "faixa_rendimento",
            "tipo_portfolio",
            "tipo_produto",
            "categoria_bem",
            "combinacao_produto",
            "setor_vendedor",
            "dia_semana_solicitacao",
            "hora_solicitacao",
            # "flag_ultima_solicitacao_contrato",
            # "flag_ultima_solicitacao_dia",
            "motivo_recusa",
            # "acompanhantes_cliente",
            # "flag_seguro_contratado"
        ]
    ).agg(
    num_max_parcelas=("numero_parcela", "max"),
    valor_total_a_pagar=("valor_parcela", "sum"),
    valor_medio_pago_parcela=("valor_pago_parcela", "mean"),
    valor_max_pago_parcela=("valor_pago_parcela", "max"),
    target=("target", "max"),
    ).reset_index()

    features_temp = features_temp.sort_values(by=["id_cliente", "data_decisao"])
    return features_temp

## Criação das Métricas Históricas

In [4]:
# importar as tabelas
cadastral = pd.read_parquet('../data/raw/base_cadastral.parquet', engine="fastparquet")
submissao = pd.read_parquet('../data/raw/base_submissao.parquet', engine="fastparquet") 
hist_contratos = pd.read_parquet('../data/raw/historico_emprestimos.parquet', engine="fastparquet")
hist_parcelas = pd.read_parquet('../data/raw/historico_parcelas.parquet', engine="fastparquet")

In [5]:
submissao.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5
1,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5
2,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5
3,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5
4,100085,2025-02-19,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5


### 1. Agregar Histórico por Contrato

In [6]:
df_hist_contract = agg_hist_contract(hist_contratos, hist_parcelas)
df_hist_contract

,id_cliente,id_contrato,tipo_contrato,status_contrato,data_decisao,valor_credito,valor_bem,valor_parcela,tipo_pagamento,finalidade_emprestimo,tipo_cliente,faixa_rendimento,tipo_portfolio,tipo_produto,categoria_bem,combinacao_produto,setor_vendedor,dia_semana_solicitacao,hora_solicitacao,motivo_recusa,num_max_parcelas,valor_total_a_pagar,valor_medio_pago_parcela,valor_max_pago_parcela,target
0,100023,1131053,Consumer loans,Approved,2018-07-07,76680.0,78705.0,8519.130,Cash through the bank,XAP,Repeater,low_normal,POS,XNA,Consumer Electronics,POS household without interest,Consumer electronics,SATURDAY,10,XAP,8.0,68153.040,10575.292500,24968.430,0
2,100023,2411919,Consumer loans,Approved,2020-02-01,93145.5,91282.5,7992.000,Cash through the bank,XAP,Repeater,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,SATURDAY,10,XAP,14.0,111888.000,9046.992857,22761.900,0
1,100023,1499902,Revolving loans,Approved,2024-04-03,45000.0,45000.0,2250.000,XNA,XAP,Refreshed,XNA,Cards,walk-in,XNA,Card Street,XNA,WEDNESDAY,11,XAP,-1.0,2250.000,-1.000000,-1.000,0
3,100023,2454202,Cash loans,Approved,2024-06-07,239242.5,180000.0,16822.440,Cash through the bank,XNA,Repeater,high,Cash,x-sell,XNA,Cash X-Sell: high,XNA,FRIDAY,9,XAP,4.0,67289.760,70571.711250,231819.525,0
4,100056,1636219,Consumer loans,Approved,2017-05-25,201528.0,223920.0,13502.385,Cash through the bank,XAP,New,high,POS,XNA,Computers,POS household with interest,Consumer electronics,THURSDAY,8,XAP,101.0,189033.390,22716.144643,168149.790,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186885,456245,1666363,Cash loans,Approved,2024-07-12,116077.5,90000.0,6481.755,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,FRIDAY,10,XAP,7.0,45372.285,6481.755000,6481.755,0
186887,456248,1395578,Consumer loans,Approved,2017-10-05,41940.0,39960.0,8417.340,Cash through the bank,XAP,New,high,POS,XNA,Mobile,POS mobile with interest,Connectivity,THURSDAY,14,XAP,6.0,84173.400,5047.740000,8417.340,0
186888,456248,1826280,Cash loans,Approved,2022-01-10,271300.5,225000.0,28633.320,Cash through the bank,XNA,Refreshed,middle,Cash,x-sell,XNA,Cash X-Sell: middle,XNA,MONDAY,13,XAP,12.0,343599.840,28608.903750,28633.320,0
186886,456248,1136073,Cash loans,Approved,2023-01-27,1333179.0,1260000.0,67725.495,Cash through the bank,XNA,Repeater,low_action,Cash,x-sell,XNA,Cash X-Sell: low,XNA,FRIDAY,16,XAP,24.0,1625411.880,67709.353125,67725.495,0


In [7]:
df_hist_contract.target.value_counts()

target
0    186054
1       836
Name: count, dtype: int64

### 2. Criar Janela Temporal Agregando as Métricas por Janela de "data_decisao"

In [8]:
# CONFIG
COLUNAS_ULTIMO_VALOR = [
    # "tipo_contrato",
    "status_contrato",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    "motivo_recusa",
    # "acompanhantes_cliente",
    # "flag_seguro_contratado",
    # "flag_ultima_solicitacao_contrato",
    # "flag_ultima_solicitacao_dia",
    "dia_semana_solicitacao",
    "hora_solicitacao",
]

AGGS = {
    "valor_credito": ["mean", "max", "min"],
    "valor_bem": ["mean", "max", "min"],
    "valor_parcela": ["mean", "max", "min"],
    # "percentual_entrada": ["mean"],
    # "qtd_parcelas_planejadas": ["max"],
    "num_max_parcelas": ["mean", "max"],
    "valor_total_a_pagar": ["mean", "sum"],
    "valor_medio_pago_parcela": ["mean"],
    "valor_max_pago_parcela": ["max"],
}

COLUNAS_REMOVER = [
    "status_contrato",
    # "percentual_entrada",
    # "qtd_parcelas_planejadas",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    # "flag_ultima_solicitacao_contrato",
    # "flag_ultima_solicitacao_dia",
    "motivo_recusa",
    # "acompanhantes_cliente",
    # "flag_seguro_contratado",
    "num_max_parcelas",
    "valor_total_a_pagar",
    "valor_medio_pago_parcela",
    "valor_max_pago_parcela",
    "hist_ultimo_dia_semana_solicitacao",
    "hist_ultimo_hora_solicitacao"
]


def create_rolling_features(df: pd.DataFrame, colunas_remover: list) -> pd.DataFrame:
    
    df = df.sort_values(["id_cliente", "data_decisao"]).copy()
    group = df.groupby("id_cliente")

    # QUANTIDADE DE CONTRATOS
    df["hist_qtd_contratos"] = group.cumcount().replace(0, np.nan)

    # FEATURES NUMÉRICAS (EXPANDING)
    for col, funcs in AGGS.items():
        tmp = (
            group[col]
            .expanding()
            .agg(funcs)
            .shift(1)
            .reset_index(level=0, drop=True)
        )

        # Caso venha como Series (apenas 1 agg)
        if isinstance(tmp, pd.Series):
            df[f"hist_{col}_{funcs[0]}"] = tmp
        else:
            for f in funcs:
                df[f"hist_{col}_{f}"] = tmp[f]

    # ÚLTIMO VALOR (SHIFT)
    for col in COLUNAS_ULTIMO_VALOR:
        df[f"hist_ultimo_{col}"] = group[col].shift(1)


    # GARANTIA: PRIMEIRA LINHA = NaN
    first_idx = group.head(1).index
    cols_hist = [c for c in df.columns if c.startswith("hist_")]

    df.loc[first_idx, cols_hist] = np.nan
    df = df.drop(columns=colunas_remover, errors="ignore")

    return df

df_window = create_rolling_features(df_hist_contract, COLUNAS_REMOVER)

In [9]:
df_window

,id_cliente,id_contrato,tipo_contrato,data_decisao,valor_credito,valor_bem,valor_parcela,dia_semana_solicitacao,hora_solicitacao,target,hist_qtd_contratos,hist_valor_credito_mean,hist_valor_credito_max,hist_valor_credito_min,hist_valor_bem_mean,hist_valor_bem_max,hist_valor_bem_min,hist_valor_parcela_mean,hist_valor_parcela_max,hist_valor_parcela_min,hist_num_max_parcelas_mean,hist_num_max_parcelas_max,hist_valor_total_a_pagar_mean,hist_valor_total_a_pagar_sum,hist_valor_medio_pago_parcela_mean,hist_valor_max_pago_parcela_max,hist_ultimo_status_contrato,hist_ultimo_tipo_pagamento,hist_ultimo_finalidade_emprestimo,hist_ultimo_tipo_cliente,hist_ultimo_faixa_rendimento,hist_ultimo_tipo_portfolio,hist_ultimo_tipo_produto,hist_ultimo_categoria_bem,hist_ultimo_combinacao_produto,hist_ultimo_setor_vendedor,hist_ultimo_motivo_recusa
0,100023,1131053,Consumer loans,2018-07-07,76680.0,78705.0,8519.130,SATURDAY,10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100023,2411919,Consumer loans,2020-02-01,93145.5,91282.5,7992.000,SATURDAY,10,0,1.0,76680.00,76680.0,76680.0,78705.00,78705.0,78705.0,8519.130,8519.130,8519.130,8.0,8.0,68153.04,68153.04,10575.292500,24968.430,Approved,Cash through the bank,XAP,Repeater,low_normal,POS,XNA,Consumer Electronics,POS household without interest,Consumer electronics,XAP
1,100023,1499902,Revolving loans,2024-04-03,45000.0,45000.0,2250.000,WEDNESDAY,11,0,2.0,84912.75,93145.5,76680.0,84993.75,91282.5,78705.0,8255.565,8519.130,7992.000,11.0,14.0,90020.52,180041.04,9811.142679,24968.430,Approved,Cash through the bank,XAP,Repeater,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,XAP
3,100023,2454202,Cash loans,2024-06-07,239242.5,180000.0,16822.440,FRIDAY,9,0,3.0,71608.50,93145.5,45000.0,71662.50,91282.5,45000.0,6253.710,8519.130,2250.000,7.0,14.0,60763.68,182291.04,6540.428452,24968.430,Approved,XNA,XAP,Refreshed,XNA,Cards,walk-in,XNA,Card Street,XNA,XAP
4,100056,1636219,Consumer loans,2017-05-25,201528.0,223920.0,13502.385,THURSDAY,8,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186885,456245,1666363,Cash loans,2024-07-12,116077.5,90000.0,6481.755,FRIDAY,10,0,1.0,47970.00,47970.0,47970.0,45000.00,45000.0,45000.0,5693.085,5693.085,5693.085,12.0,12.0,68317.02,68317.02,5692.650000,5693.085,Approved,Cash through the bank,Urgent needs,New,high,Cash,walk-in,XNA,Cash Street: high,Connectivity,XAP
186887,456248,1395578,Consumer loans,2017-10-05,41940.0,39960.0,8417.340,THURSDAY,14,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186888,456248,1826280,Cash loans,2022-01-10,271300.5,225000.0,28633.320,MONDAY,13,0,1.0,41940.00,41940.0,41940.0,39960.00,39960.0,39960.0,8417.340,8417.340,8417.340,6.0,6.0,84173.40,84173.40,5047.740000,8417.340,Approved,Cash through the bank,XAP,New,high,POS,XNA,Mobile,POS mobile with interest,Connectivity,XAP
186886,456248,1136073,Cash loans,2023-01-27,1333179.0,1260000.0,67725.495,FRIDAY,16,0,2.0,156620.25,271300.5,41940.0,132480.00,225000.0,39960.0,18525.330,28633.320,8417.340,9.0,12.0,213886.62,427773.24,16828.321875,28633.320,Approved,Cash through the bank,XNA,Refreshed,middle,Cash,x-sell,XNA,Cash X-Sell: middle,XNA,XAP


### 3. Unir dados de janela com os dados cadastrais

In [10]:
df_window_final = df_window.merge(
    cadastral, 
    how="left", 
    on="id_cliente"
)

df_window_final = df_window_final.drop(["sexo", "data_nascimento"], axis=1)

### 4. Criar tabelas de treino e teste

In [11]:
df_window_final = df_window_final.sort_values(
    ["id_cliente", "data_decisao", "hist_qtd_contratos", "id_contrato"]
)

In [12]:
df_test = df_window_final.groupby("id_cliente").tail(1).copy()

df_test = df_test.drop([
    "tipo_contrato",
    "valor_credito",
    "valor_bem",
    "valor_parcela",
    "dia_semana_solicitacao",
    "hora_solicitacao",
    "target",
    "id_contrato",
    "data_decisao"
], axis=1)

df_test = submissao.merge(df_test, how="left", on="id_cliente")

df_test = df_test.drop([
    "data_solicitacao"
], axis=1)

df_test = df_test.replace(-1, np.nan)

In [13]:
df_test

,id_cliente,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela,hist_qtd_contratos,hist_valor_credito_mean,hist_valor_credito_max,hist_valor_credito_min,hist_valor_bem_mean,hist_valor_bem_max,hist_valor_bem_min,hist_valor_parcela_mean,hist_valor_parcela_max,hist_valor_parcela_min,hist_num_max_parcelas_mean,hist_num_max_parcelas_max,hist_valor_total_a_pagar_mean,hist_valor_total_a_pagar_sum,hist_valor_medio_pago_parcela_mean,hist_valor_max_pago_parcela_max,hist_ultimo_status_contrato,hist_ultimo_tipo_pagamento,hist_ultimo_finalidade_emprestimo,hist_ultimo_tipo_cliente,hist_ultimo_faixa_rendimento,hist_ultimo_tipo_portfolio,hist_ultimo_tipo_produto,hist_ultimo_categoria_bem,hist_ultimo_combinacao_produto,hist_ultimo_setor_vendedor,hist_ultimo_motivo_recusa,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
0,100023,MONDAY,12,Cash loans,544491.0,454500.0,17563.5,3.0,71608.5000,93145.5,45000.0,71662.500000,91282.5,45000.0,6253.710000,8519.130,2250.000,7.000000,14.0,60763.680000,182291.040,6540.428452,24968.430,Approved,XNA,XAP,Refreshed,XNA,Cards,walk-in,XNA,Card Street,XNA,XAP,1.0,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2.0,2.0
1,100031,MONDAY,9,Cash loans,979992.0,702000.0,27076.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100056,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5,2.0,171976.5000,201528.0,142425.0,179460.000000,223920.0,135000.0,15503.220000,17504.055,13502.385,53.000000,101.0,138276.832500,276553.665,28458.247821,168149.790,Approved,XNA,XNA,Repeater,middle,Cash,x-sell,XNA,Cash Street: middle,Consumer electronics,XAP,0.0,2.0,360000.0,Working,Laborers,Transport: type 2,Secondary / secondary special,Married,House / apartment,Y,Y,2.0,2.0
3,100069,MONDAY,11,Cash loans,640458.0,517500.0,27265.5,6.0,146900.2500,317925.0,0.0,150561.695833,353250.0,NaN,16489.865833,30174.435,NaN,9.000000,12.0,181512.913333,1089077.480,15468.677530,30174.435,Approved,Cash through the bank,XNA,Repeater,middle,Cash,x-sell,XNA,Cash X-Sell: middle,XNA,XAP,1.0,2.0,360000.0,Working,Laborers,Transport: type 4,Secondary / secondary special,Separated,House / apartment,Y,Y,2.0,2.0
4,100085,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5,2.0,22916.2500,23985.0,21847.5,26291.250000,28597.5,23985.0,2274.632500,4550.265,NaN,2.500000,6.0,13650.295000,27300.590,2274.295000,4550.265,Refused,Cash through the bank,XAP,Repeater,XNA,XNA,XNA,Mobile,POS mobile with interest,Connectivity,HC,1.0,3.0,157500.0,Working,Drivers,Business Entity Type 1,Secondary / secondary special,Married,House / apartment,N,Y,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,455962,WEDNESDAY,12,Cash loans,156384.0,135000.0,16551.0,1.0,80955.0000,80955.0,80955.0,80955.000000,80955.0,80955.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unused offer,Cash through the bank,XAP,Refreshed,XNA,XNA,XNA,Computers,POS mobile with interest,Connectivity,CLIENT,0.0,2.0,112500.0,Working,Laborers,Security,Higher education,Married,House / apartment,N,Y,2.0,2.0
39996,456007,MONDAY,10,Cash loans,318645.0,216000.0,16398.0,4.0,179615.2500,381528.0,93474.0,157460.625000,315000.0,89424.0,18670.061250,24841.305,15359.130,6.250000,12.0,126837.146250,507348.585,14540.444375,24841.305,Approved,Cash through the bank,XNA,Repeater,middle,Cash,x-sell,XNA,Cash X-Sell: middle,XNA,XAP,0.0,1.0,135000.0,Working,Drivers,Business Entity Type 3,Secondary / secondary special,Single / not married,House / apartment,N,Y,2.0,2.0
39997,456009,TUESDAY,14,Cash loans,360000.0,360000.0,19530.0,8.0,146858.0625,343377.0,0

In [14]:
df_train = df_window_final.drop(df_test.index)
df_train = df_train.drop([
    "id_cliente",
    "id_contrato",
    "data_decisao"
], axis=1)

df_train = df_train.replace(-1, np.nan)

In [15]:
df_train

,tipo_contrato,valor_credito,valor_bem,valor_parcela,dia_semana_solicitacao,hora_solicitacao,target,hist_qtd_contratos,hist_valor_credito_mean,hist_valor_credito_max,hist_valor_credito_min,hist_valor_bem_mean,hist_valor_bem_max,hist_valor_bem_min,hist_valor_parcela_mean,hist_valor_parcela_max,hist_valor_parcela_min,hist_num_max_parcelas_mean,hist_num_max_parcelas_max,hist_valor_total_a_pagar_mean,hist_valor_total_a_pagar_sum,hist_valor_medio_pago_parcela_mean,hist_valor_max_pago_parcela_max,hist_ultimo_status_contrato,hist_ultimo_tipo_pagamento,hist_ultimo_finalidade_emprestimo,hist_ultimo_tipo_cliente,hist_ultimo_faixa_rendimento,hist_ultimo_tipo_portfolio,hist_ultimo_tipo_produto,hist_ultimo_categoria_bem,hist_ultimo_combinacao_produto,hist_ultimo_setor_vendedor,hist_ultimo_motivo_recusa,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
40000,Cash loans,1005120.0,720000.0,29520.225,SUNDAY,4,0,9.0,465248.00,843552.0,0.0,358999.555556,720000.0,NaN,16311.631667,32272.605,NaN,6.555556,49.0,183020.996667,1647188.970,8232.007118,649299.330,Canceled,XNA,XNA,Repeater,XNA,XNA,XNA,XNA,Cash,XNA,XAP,0,1.0,189000.0,Working,None,Business Entity Type 3,Secondary / secondary special,Single / not married,House / apartment,N,Y,2,2
40001,Cash loans,0.0,NaN,NaN,SUNDAY,4,0,10.0,519235.20,1005120.0,0.0,395099.600000,720000.0,NaN,17632.491000,32272.605,NaN,5.800000,49.0,167670.919500,1676709.195,7408.706406,649299.330,Refused,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,HC,0,1.0,189000.0,Working,None,Business Entity Type 3,Secondary / secondary special,Single / not married,House / apartment,N,Y,2,2
40002,Consumer loans,72976.5,66006.0,7332.660,THURSDAY,15,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3.0,162000.0,Working,Security staff,Other,Incomplete higher,Married,House / apartment,Y,Y,2,2
40003,Consumer loans,140211.0,115762.5,6350.175,FRIDAY,8,0,1.0,72976.50,72976.5,72976.5,66006.000000,66006.0,66006.0,7332.660000,7332.660,7332.660,12.000000,12.0,87991.920000,87991.920,7329.585000,7332.660,Approved,Cash through the bank,XAP,New,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,XAP,1,3.0,162000.0,Working,Security staff,Other,Incomplete higher,Married,House / apartment,Y,Y,2,2
40004,Cash loans,0.0,NaN,NaN,TUESDAY,8,0,2.0,106593.75,140211.0,72976.5,90884.250000,115762.5,66006.0,6841.417500,7332.660,6350.175,13.000000,14.0,94797.360000,189594.720,7960.239375,30798.000,Approved,Cash through the bank,XAP,Repeater,low_action,POS,XNA,Computers,POS household without interest,Consumer electronics,XAP,1,3.0,162000.0,Working,Security staff,Other,Incomplete higher,Married,House / apartment,Y,Y,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186885,Cash loans,116077.5,90000.0,6481.755,FRIDAY,10,0,1.0,47970.00,47970.0,47970.0,45000.000000,45000.0,45000.0,5693.085000,5693.085,5693.085,12.000000,12.0,68317.020000,68317.020,5692.650000,5693.085,Approved,Cash through the bank,Urgent needs,New,high,Cash,walk-in,XNA,Cash Street: high,Connectivity,XAP,3,5.0,81000.0,Commercial associate,Low-skill Laborers,Industry: type 1,Secondary / secondary special,Married,House / apartment,N,Y,2,2
186886,Consumer loans,41940.0,39960.0,8417.340,THURSDAY,14,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1.0,153000.0,Working,Sales staff,Self-employed,Secondary / secondary special,Separated,House / apartment,N,Y,2,2
186887,Cash loans,271300.5,225000.0,28633.320,MONDAY,13,0,1.0,41940.00,41940.0,41940.0,39960.000000,39960.0,39960.0,8417.340000,8417.340,8417.340,6.000000,6.0,84173.400000,84173.400

In [16]:
output_dir = "../data/curated"
os.makedirs(output_dir, exist_ok=True)
df_train.to_parquet(f"{output_dir}/train.parquet", index=False)
df_test.to_parquet(f"{output_dir}/test.parquet", index=False)